In [1]:
from gam_rs_utils.utils import *
from src.utils import binary_to_one_hot
from src.prepare_gam import get_fastsparse
from src.rset_opt import RSetOPT
from src.run_app import get_models_from_rset

In [ ]:
dn = 'bank'
data = pd.read_csv(f'datasets/{dn}.csv')
l0 = 0.001
l2 = 0.001
m = 1.01

w, y, header, X_orig = get_fastsparse(data, l0, l2)


print("data shape", data.shape)
print("w", len(w), "y", len(y), "header", len(header))
print("X_new.shape", X_new.shape, "header_new", header_new)

data shape (4521, 17)
w 3719 y 4521 header 3719
X_new.shape (4521, 21) header_new ['intercept', 'housing<=0.0', '0.0<housing<=1', 'loan<=0.0', '0.0<loan<=1', 'contact<=1.0', '1.0<contact<=2', 'month<=9.0', '9.0<month<=11', 'duration<=211.0', '211.0<duration<=348.0', '348.0<duration<=645.0', '645.0<duration<=770.0', '770.0<duration<=3025', 'pdays<=374.0', '374.0<pdays<=871', 'previous<=0.0', '0.0<previous<=1.0', '1.0<previous<=25', 'poutcome<=1.0', '1.0<poutcome<=3']


In [10]:
data['month'].min(), data['month'].max()

(0, 11)

In [ ]:
from FasterRisk.src.fasterrisk import fasterrisk

start = time()
        
y = y.ravel()
X_new, header_new = utils.binary_to_one_hot(data.iloc[:,:-1], w, header)

rs = fasterrisk.RiskScoreOptimizer(
    X_one_hot_no_intercept, y, 
    k=n_support_set, 
    lb=-100, ub=100, 
    gap_tolerance=m - 1.0, 
    select_top_m=-1, 
    maxAttempts=25
)
beam_size = 100
rs.optimize_with_swaps_beam_search(
    swaps=k, 
    beam_size=beam_size, 
    verbose=True, 
    beta0=w_orig_zeroed[0], 
    betas=w_orig_zeroed[1:]
)

end = time()